In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

df_bra   = spark.table("silver.brasileirao")
df_times = spark.table("silver.times")
df_est   = spark.table("silver.estadio")

# ---- DIM DATA ----
dim_data = (
    df_bra.select("data").where(F.col("data").isNotNull()).dropDuplicates(["data"])
    .withColumn("data_id", F.date_format("data", "yyyyMMdd").cast("int"))
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("dia_semana", F.date_format("data", "E"))
)

(dim_data.write.format("delta").mode("overwrite").option("overwriteSchema","true")
.saveAsTable("gold.dim_data"))

# ---- DIM TIME ----
dim_time = (
    df_times
    .withColumn("time_id", F.sha2(F.col("time"), 256))
    .select("time_id", "time", "apelido")
)

(dim_time.write.format("delta").mode("overwrite").option("overwriteSchema","true")
.saveAsTable("gold.dim_time"))

# ---- DIM ESTADIO ----
dim_estadio = (
    df_est
    .withColumn("estadio_id", F.sha2(F.col("estadio"), 256))
    .select("estadio_id", "estadio", "cidade", "capacidade_maxima")
)

(dim_estadio.write.format("delta").mode("overwrite").option("overwriteSchema","true")
.saveAsTable("gold.dim_estadio"))

# ---- FACT PARTIDA ----
dim_time_tbl = spark.table("gold.dim_time")
dim_est_tbl  = spark.table("gold.dim_estadio")
dim_data_tbl = spark.table("gold.dim_data")

fact = (
    df_bra
    .join(dim_data_tbl.select("data","data_id"), on="data", how="left")
    .join(dim_time_tbl.select(F.col("time").alias("tm"), F.col("time_id").alias("time_mandante_id")),
          df_bra["time_mandante"] == F.col("tm"), "left")
    .join(dim_time_tbl.select(F.col("time").alias("tv"), F.col("time_id").alias("time_visitante_id")),
          df_bra["time_visitante"] == F.col("tv"), "left")
    .join(dim_est_tbl.select(F.col("estadio").alias("e"), "estadio_id"),
          df_bra["estadio"] == F.col("e"), "left")
)

fact_partida = (
    fact
    .withColumn("partida_id", F.sha2(F.concat_ws("||",
                                                F.col("ano_campeonato").cast("string"),
                                                F.col("data").cast("string"),
                                                F.col("rodada").cast("string"),
                                                F.col("time_mandante_id"),
                                                F.col("time_visitante_id")), 256))
    .select(
        "partida_id",
        "ano_campeonato",
        "data_id",
        "rodada",
        "estadio_id",
        "time_mandante_id",
        "time_visitante_id",
        "gols_mandante",
        "gols_visitante",
        "publico",
        "publico_max"
    )
)

(fact_partida.write.format("delta").mode("overwrite").option("overwriteSchema","true")
.saveAsTable("gold.fact_partida"))

In [0]:
spark.table("silver.brasileirao").select("data").printSchema()